# Extract Documents

Download or read the indexed PDFs and produce canonical text-extraction manifests and document JSON files.

**Requires:** `scan/included.index.json`; Google credentials only for Drive-backed files.  
**Produces:** document manifests, document JSON files, indexes, and an extraction report.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd().resolve()
while not (repo_root / "pipeline.json").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if not (repo_root / "pipeline.json").exists():
    raise FileNotFoundError("Open this notebook from inside the Nursind repository")

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from IPython.display import display
from notebooks import interface as nb
from notebooks.shared_config import load_notebook_context

ctx = load_notebook_context(repo_root / "pipeline.json")
paths = ctx.paths
step_cfg = ctx.step("extract_documents")
display(nb.pipeline_overview(ctx, "extract_documents"))


## Controls


In [ ]:
VERBOSE = True
REPROCESS_INCLUDED = False
REPROCESS_EXCLUDED = False
LIMIT = int(step_cfg.get("limit", 0))  # 0 means all documents

{
    "reprocess_included": REPROCESS_INCLUDED,
    "reprocess_excluded": REPROCESS_EXCLUDED,
    "limit": LIMIT,
}


## Input


In [ ]:
display(nb.artifact_table({"scan index": paths.scan_included_index}))
scan_index = nb.preview_json(paths.scan_included_index)
{
    "employee_count": scan_index.get("employee_count"),
    "total_files": scan_index.get("total_files"),
}


## Build Options


In [ ]:
from core.documents.options import ExtractDocumentsFromIndexOptions

options = ExtractDocumentsFromIndexOptions(
    out=str(paths.documents_dir),
    index=str(paths.scan_included_index),
    included=str(paths.documents_included_index),
    excluded=str(paths.documents_excluded_index),
    report=str(paths.documents_report),
    reprocess_included=REPROCESS_INCLUDED,
    reprocess_excluded=REPROCESS_EXCLUDED,
    workers=int(step_cfg.get("workers", 8)),
    download_workers=step_cfg.get("download_workers"),
    extract_workers=int(step_cfg.get("extract_workers", 1)),
    max_in_flight=int(step_cfg.get("max_in_flight", 128)),
    flush_every=int(step_cfg.get("flush_every", 100)),
    limit=LIMIT,
    log_every=int(step_cfg.get("log_every", 50)),
    min_normal_score=float(step_cfg.get("min_normal_score", 0.72)),
    min_score_delta=float(step_cfg.get("min_score_delta", 0.08)),
    verbose=VERBOSE,
)
options


## Run Extraction


In [ ]:
from core.documents.runtime import run_extraction
from core.drive.logging_utils import setup_logging

setup_logging(VERBOSE)
exit_code = run_extraction(options, configure_logging=False)
document_report = nb.preview_json(paths.documents_report)
display(nb.report_summary(document_report))
{"exit_code": exit_code}


## Inspect Results


In [ ]:
display(nb.artifact_table({
    "included documents": paths.documents_included_index,
    "excluded documents": paths.documents_excluded_index,
    "extraction report": paths.documents_report,
}))
display(nb.file_table(paths.documents_dir / "docs", "*.json"))
